# 🚗 Object Detection & Tracking with YOLO11 on BDD100K
**Run this notebook top-to-bottom on Google Colab (GPU runtime)**

### 📁 Your Dataset Location (Local PC):
```
C:\Users\Shivani Agarwal\Downloads\Garv_minor2\Object\10k\
    images\  (train / val / test)
    labels\  (train / val / test)

C:\Users\Shivani Agarwal\Downloads\Garv_minor2\Object\6.mp4
```

### ✅ Two Ways to Load Dataset into Colab:
- **Option A** *(Recommended for large datasets)*: Mount Google Drive
- **Option B** *(Quick)*: Upload a `.zip` directly from your PC

**Run only ONE of Option A or Option B below, then continue from Step 2.**

---
## Option A — Google Drive Mount
Upload your `10k/` folder to `MyDrive/10k/` on Google Drive, and `6.mp4` to `MyDrive/6.mp4`, then run this cell.

In [1]:
# ── OPTION A: Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# Upload '10k' folder and '6.mp4' directly to MyDrive root
DATASET_ROOT = '/content/drive/MyDrive/10k'
VIDEO_PATH   = '/content/drive/MyDrive/6.mp4'

assert os.path.isdir(DATASET_ROOT), (
    f'❌ Not found: {DATASET_ROOT}\n'
    '   Upload the 10k/ folder to the ROOT of your Google Drive (MyDrive/10k/)'
)
assert os.path.isfile(VIDEO_PATH), (
    f'❌ Not found: {VIDEO_PATH}\n'
    '   Upload 6.mp4 to the ROOT of your Google Drive (MyDrive/6.mp4)'
)

print('✅ Drive mounted!')
print(f'   Dataset : {DATASET_ROOT}')
print(f'   Video   : {VIDEO_PATH}')

ModuleNotFoundError: No module named 'google'

---
## Option B — Upload Directly from Your PC (Zip Method)

**Step 1:** Zip your dataset on your PC (open PowerShell):
```powershell
Compress-Archive -Path "C:\Users\Shivani Agarwal\Downloads\Garv_minor2\Object\10k" -DestinationPath "C:\Users\Shivani Agarwal\Desktop\10k.zip"
```
**Step 2:** Run the cell below → it will open a file picker → select `10k.zip` and `6.mp4` from your PC.

In [2]:
# ── OPTION B: Direct Upload from PC ──────────────────────────────────────
from google.colab import files
import zipfile, os

print('📤 Select your 10k.zip file (created from your 10k/ folder)...')
uploaded = files.upload()   # ← opens file picker

# Find the uploaded zip
zip_name = [f for f in uploaded.keys() if f.endswith('.zip')]
assert zip_name, '❌ No .zip file found in upload!'
zip_name = zip_name[0]

print(f'\n📦 Extracting {zip_name} ...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

DATASET_ROOT = '/content/10k'
assert os.path.isdir(DATASET_ROOT), \
    f'❌ Extraction failed — expected folder at {DATASET_ROOT}'
print(f'✅ Dataset extracted to {DATASET_ROOT}')

print('\n📤 Now select your 6.mp4 video file...')
video_up = files.upload()
vid_name = [f for f in video_up.keys() if f.endswith('.mp4')]
assert vid_name, '❌ No .mp4 file found!'

VIDEO_PATH = f'/content/{vid_name[0]}'
print(f'✅ Video ready at {VIDEO_PATH}')

ModuleNotFoundError: No module named 'google'

---
## Step 2 — Install Dependencies

In [3]:
!pip install -q ultralytics opencv-python-headless

import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


PyTorch version : 2.9.1+cpu
CUDA available  : False
⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU


## Step 3 — Create `data.yaml`

In [4]:
import yaml, os

print(f'Dataset root: {DATASET_ROOT}')
print('\nChecking sub-folders...')
for split in ['train', 'val', 'test']:
    img_p = os.path.join(DATASET_ROOT, 'images', split)
    lbl_p = os.path.join(DATASET_ROOT, 'labels', split)
    assert os.path.isdir(img_p), f'Missing: {img_p}'
    assert os.path.isdir(lbl_p), f'Missing: {lbl_p}'
    n_img = len(os.listdir(img_p))
    n_lbl = len(os.listdir(lbl_p))
    print(f'  {split:5s} → {n_img:5d} images | {n_lbl:5d} labels')

# BDD100K 6-class subset
CLASSES = [
    'pedestrian',   # 0
    'rider',        # 1
    'car',          # 2
    'truck',        # 3
    'bus',          # 4
    'motorcycle',   # 5
]

data_yaml_path = '/content/data.yaml'
data_dict = {
    'path' : DATASET_ROOT,
    'train': 'images/train',
    'val'  : 'images/val',
    'test' : 'images/test',
    'nc'   : len(CLASSES),
    'names': CLASSES,
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_dict, f, default_flow_style=False, sort_keys=False)

print(f'\n✅ data.yaml created')
print(open(data_yaml_path).read())

NameError: name 'DATASET_ROOT' is not defined

## Step 4 — Train YOLO11n

In [ ]:
from ultralytics import YOLO
import time

model = YOLO('yolo11n.pt')

t0 = time.time()
model.train(
    data     = data_yaml_path,
    epochs   = 10,           # ← increase to 50–100 for better accuracy
    imgsz    = 640,
    batch    = 16,           # ← reduce to 8 if OOM error appears
    device   = 0,
    project  = '/content/checkpoints',
    name     = 'yolo_bdd100k',
    workers  = 2,
    exist_ok = True,
    verbose  = True,
)

print(f'\n✅ Training done in {(time.time()-t0)/60:.1f} min')
BEST_WEIGHTS = '/content/checkpoints/yolo_bdd100k/weights/best.pt'
print('Best weights:', BEST_WEIGHTS)

## Step 5 — Validate / Evaluate

In [ ]:
from ultralytics import YOLO
import time

eval_model = YOLO(BEST_WEIGHTS)

t0 = time.time()
metrics = eval_model.val(data=data_yaml_path, device=0)
t1 = time.time()

print('\n=== Evaluation Metrics ===')
print(f'Precision  : {metrics.box.mp:.4f}')
print(f'Recall     : {metrics.box.mr:.4f}')
print(f'mAP@0.50   : {metrics.box.map50:.4f}')
print(f'mAP@0.5:95 : {metrics.box.map:.4f}')
inf_ms = metrics.speed.get('inference', 0)
print(f'FPS        : {1000/inf_ms:.1f}' if inf_ms > 0 else 'FPS: N/A')
print(f'Val time   : {t1-t0:.1f} s')

## Step 6 — 📊 Plot Metrics Graphs
Generates bar chart + per-epoch training curves.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
import os

C_PREC  = '#4C72B0'
C_REC   = '#55A868'
C_MAP50 = '#C44E52'
C_MAP95 = '#8172B2'
C_FPS   = '#CCB974'
PALETTE = [C_PREC, C_REC, C_MAP50, C_MAP95, C_FPS]

precision = metrics.box.mp
recall    = metrics.box.mr
map50     = metrics.box.map50
map5095   = metrics.box.map
inf_ms    = metrics.speed.get('inference', 1e-9)
fps_val   = round(1000 / inf_ms, 1) if inf_ms > 0 else 0

# ════════════════════════════════
# FIGURE 1 — Summary Bar Chart
# ════════════════════════════════
labels_bar = ['Precision', 'Recall', 'mAP\n@0.5', 'mAP\n@0.5:95', 'FPS\n(÷100)']
fps_norm   = min(fps_val / 100, 1.0)
values_bar = [precision, recall, map50, map5095, fps_norm]
raw_values = [precision, recall, map50, map5095, fps_val]

fig1, ax1 = plt.subplots(figsize=(11, 5.5))
fig1.patch.set_facecolor('#FAFAFA')
ax1.set_facecolor('#FAFAFA')

bars = ax1.bar(np.arange(len(labels_bar)), values_bar,
               color=PALETTE, edgecolor='white', linewidth=1.4,
               width=0.52, zorder=3)

for bar, raw in zip(bars, raw_values):
    txt = f'{raw:.3f}' if raw < 10 else f'{raw:.1f}'
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.013,
             txt, ha='center', va='bottom', fontsize=12,
             fontweight='bold', color='#1a1a1a')

ax1.set_xticks(np.arange(len(labels_bar)))
ax1.set_xticklabels(labels_bar, fontsize=12)
ax1.set_ylim(0, 1.18)
ax1.set_ylabel('Score  (FPS normalised ÷100)', fontsize=11)
ax1.set_title('YOLO11 — Validation Metrics Summary (BDD100K)',
              fontsize=14, fontweight='bold', pad=16)
ax1.yaxis.grid(True, linestyle='--', alpha=0.45, zorder=0)
ax1.set_axisbelow(True)
ax1.spines[['top', 'right']].set_visible(False)
ax1.legend(
    handles=[mpatches.Patch(color=C_FPS,
        label=f'FPS actual = {fps_val}  (bar = FPS÷100)')],
    fontsize=9, loc='upper right'
)

plt.tight_layout()
plt.savefig('/content/metrics_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 1 saved: /content/metrics_bar.png')

# ════════════════════════════════════
# FIGURE 2 — Per-Epoch Training Curves
# ════════════════════════════════════
RESULTS_CSV = '/content/checkpoints/yolo_bdd100k/results.csv'

if os.path.isfile(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV)
    df.columns = df.columns.str.strip()

    df.rename(columns={
        'metrics/precision(B)' : 'precision',
        'metrics/recall(B)'    : 'recall',
        'metrics/mAP50(B)'     : 'map50',
        'metrics/mAP50-95(B)'  : 'map5095',
        'train/box_loss'       : 'box_loss',
        'train/cls_loss'       : 'cls_loss',
        'train/dfl_loss'       : 'dfl_loss',
        'val/box_loss'         : 'val_box_loss',
        'val/cls_loss'         : 'val_cls_loss',
    }, inplace=True)

    epochs = (df['epoch'].values if 'epoch' in df.columns
              else np.arange(1, len(df)+1))
    LINE   = dict(linewidth=2.2, marker='o', markersize=5)

    fig2 = plt.figure(figsize=(15, 10), facecolor='#FAFAFA')
    fig2.suptitle('YOLO11 — Per-Epoch Training Curves (BDD100K)',
                  fontsize=16, fontweight='bold', y=1.01)
    gs = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.32)

    # Panel A — Precision & Recall
    ax_a = fig2.add_subplot(gs[0, 0])
    ax_a.set_facecolor('#FAFAFA')
    if 'precision' in df.columns:
        ax_a.plot(epochs, df['precision'], color=C_PREC, label='Precision', **LINE)
    if 'recall' in df.columns:
        ax_a.plot(epochs, df['recall'], color=C_REC, label='Recall', **LINE)
    ax_a.set_title('Precision & Recall', fontweight='bold', fontsize=12)
    ax_a.set_xlabel('Epoch'); ax_a.set_ylabel('Score')
    ax_a.set_ylim(0, 1.05); ax_a.legend(fontsize=10)
    ax_a.grid(True, linestyle='--', alpha=0.4)
    ax_a.spines[['top', 'right']].set_visible(False)

    # Panel B — mAP Curves
    ax_b = fig2.add_subplot(gs[0, 1])
    ax_b.set_facecolor('#FAFAFA')
    if 'map50' in df.columns:
        ax_b.plot(epochs, df['map50'], color=C_MAP50, label='mAP@0.5', **LINE)
    if 'map5095' in df.columns:
        ax_b.plot(epochs, df['map5095'], color=C_MAP95, label='mAP@0.5:95', **LINE)
    ax_b.set_title('mAP Curves', fontweight='bold', fontsize=12)
    ax_b.set_xlabel('Epoch'); ax_b.set_ylabel('mAP')
    ax_b.set_ylim(0, 1.05); ax_b.legend(fontsize=10)
    ax_b.grid(True, linestyle='--', alpha=0.4)
    ax_b.spines[['top', 'right']].set_visible(False)

    # Panel C — Losses
    ax_c = fig2.add_subplot(gs[1, 0])
    ax_c.set_facecolor('#FAFAFA')
    for col, clr, ls, lbl in [
        ('box_loss',     '#D65F5F', '-',  'Box Loss (train)'),
        ('cls_loss',     '#EE8866', '-',  'Cls Loss (train)'),
        ('dfl_loss',     '#44BB99', '-',  'DFL Loss (train)'),
        ('val_box_loss', '#D65F5F', '--', 'Box Loss (val)'),
        ('val_cls_loss', '#EE8866', '--', 'Cls Loss (val)'),
    ]:
        if col in df.columns:
            ax_c.plot(epochs, df[col], color=clr, linestyle=ls,
                      label=lbl, linewidth=2.2, marker='o', markersize=4)
    ax_c.set_title('Training & Validation Losses', fontweight='bold', fontsize=12)
    ax_c.set_xlabel('Epoch'); ax_c.set_ylabel('Loss')
    ax_c.legend(fontsize=9)
    ax_c.grid(True, linestyle='--', alpha=0.4)
    ax_c.spines[['top', 'right']].set_visible(False)

    # Panel D — FPS Gauge
    ax_d = fig2.add_subplot(gs[1, 1])
    ax_d.set_facecolor('#F2F2F2'); ax_d.axis('off')
    fps_color  = '#27AE60' if fps_val>=30 else ('#F39C12' if fps_val>=15 else '#E74C3C')
    fps_status = ('🟢 Real-time capable' if fps_val>=30
                  else ('🟡 Near real-time' if fps_val>=15 else '🔴 Below real-time'))
    ax_d.text(0.5, 0.72, f'{fps_val}', transform=ax_d.transAxes,
              ha='center', va='center', fontsize=64,
              fontweight='bold', color=fps_color)
    ax_d.text(0.5, 0.46, 'Frames Per Second',
              transform=ax_d.transAxes, ha='center', fontsize=13, color='#444')
    ax_d.text(0.5, 0.30, f'({inf_ms:.2f} ms / image)',
              transform=ax_d.transAxes, ha='center', fontsize=11, color='#666')
    ax_d.text(0.5, 0.13, fps_status, transform=ax_d.transAxes,
              ha='center', fontsize=13, fontweight='bold', color=fps_color)
    ax_d.set_title('Inference Speed (Validation)', fontweight='bold', fontsize=12)

    plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Figure 2 saved: /content/training_curves.png')
else:
    print('⚠️  results.csv not found — run Step 4 (training) first.')

## Step 7 — Vehicle Tracking + Counting on Video

In [ ]:
import math

class Tracker:
    def __init__(self):
        self.center_points = {}
        self.id_count = 0

    def update(self, objects_rect):
        objects_bbs_ids = []
        for rect in objects_rect:
            x1, y1, x2, y2 = rect
            cx = (x1 + x2) // 2   # FIXED
            cy = (y1 + y2) // 2
            same = False
            for oid, pt in self.center_points.items():
                if math.hypot(cx-pt[0], cy-pt[1]) < 50:
                    self.center_points[oid] = (cx, cy)
                    objects_bbs_ids.append([x1,y1,x2,y2,oid])
                    same = True; break
            if not same:
                self.center_points[self.id_count] = (cx, cy)
                objects_bbs_ids.append([x1,y1,x2,y2,self.id_count])
                self.id_count += 1
        ncp = {}
        for *_, oid in objects_bbs_ids:
            ncp[oid] = self.center_points[oid]
        self.center_points = ncp.copy()
        return objects_bbs_ids

print('✅ Tracker ready.')

In [ ]:
import cv2, time, numpy as np
from ultralytics import YOLO

VIDEO_OUT   = '/content/output_tracked.mp4'
CONF_THRESH = 0.35
LINE_Y_RED  = 200
LINE_Y_BLUE = 350

detect_model = YOLO(BEST_WEIGHTS)
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f'Cannot open: {VIDEO_PATH}'

W  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FS = cap.get(cv2.CAP_PROP_FPS) or 30
TF = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {W}×{H} @ {FS:.1f} FPS | {TF} frames')

writer = cv2.VideoWriter(VIDEO_OUT, cv2.VideoWriter_fourcc(*'mp4v'), FS, (W,H))

prev_pos,speeds = {},{}
cdn,cup = set(),set()
tdn=tup=fn=0
t0=time.time()
CN = detect_model.names

while True:
    ret,frame = cap.read()
    if not ret: break
    fn += 1

    res = detect_model.track(frame, persist=True, tracker='bytetrack.yaml',
                             conf=CONF_THRESH, verbose=False)

    if res and res[0].boxes is not None and res[0].boxes.id is not None:
        boxes   = res[0].boxes.xyxy.cpu().numpy().astype(int)
        ids     = res[0].boxes.id.cpu().numpy().astype(int)
        cls_ids = res[0].boxes.cls.cpu().numpy().astype(int)

        for box,oid,cid in zip(boxes,ids,cls_ids):
            x1,y1,x2,y2 = box
            cx,cy = (x1+x2)//2,(y1+y2)//2
            lbl = CN.get(cid,str(cid))

            if oid in prev_pos:
                px,py = prev_pos[oid]
                speeds[oid] = float(np.hypot(cx-px,cy-py))
            else: speeds[oid] = 0.0
            prev_pos[oid] = (cx,cy)

            M=6
            if LINE_Y_RED-M<=cy<=LINE_Y_RED+M:
                cdn.add(oid)
                if oid in cup: tup+=1; cup.discard(oid)
            if LINE_Y_BLUE-M<=cy<=LINE_Y_BLUE+M:
                cup.add(oid)
                if oid in cdn: tdn+=1; cdn.discard(oid)

            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.circle(frame,(cx,cy),4,(0,0,255),-1)
            cv2.putText(frame,f'{lbl} ID:{oid} Spd:{speeds[oid]:.1f}',
                        (x1,max(y1-8,12)),cv2.FONT_HERSHEY_SIMPLEX,0.45,(255,255,255),1)

    cv2.line(frame,(0,LINE_Y_RED),(W,LINE_Y_RED),(0,0,255),2)
    cv2.line(frame,(0,LINE_Y_BLUE),(W,LINE_Y_BLUE),(255,0,0),2)
    fps_live = fn/(time.time()-t0)
    for txt,y in [(f'Down:{tdn}',30),(f'Up:{tup}',60),(f'FPS:{fps_live:.1f}',90)]:
        cv2.putText(frame,txt,(10,y),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

    writer.write(frame)
    if fn%50==0: print(f'  Frame {fn}/{TF} | Down={tdn} Up={tup}')

cap.release(); writer.release()
print(f'\n✅ Done | Down={tdn} | Up={tup} | Saved: {VIDEO_OUT}')

## Step 8 — Preview Output Video

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def show_video(path, width=800):
    b64 = b64encode(open(path,'rb').read()).decode()
    return HTML(f"<video width='{width}' controls>"
                f"<source src='data:video/mp4;base64,{b64}' type='video/mp4'>"
                f"</video>")

show_video('/content/output_tracked.mp4')

## Step 9 — Download Outputs to Your PC

In [ ]:
from google.colab import files
import os

# Download best weights
files.download(BEST_WEIGHTS)
print('✅ best.pt downloading...')

# Download output video
files.download('/content/output_tracked.mp4')
print('✅ output_tracked.mp4 downloading...')

# Download graphs
for g in ['metrics_bar.png', 'training_curves.png']:
    if os.path.isfile(f'/content/{g}'):
        files.download(f'/content/{g}')
        print(f'✅ {g} downloading...')